# Genshin 对话数据清洗与导入 MongoDB

1. 从本地 JSON 加载数据  
2. 保留 `page_name`、`component_name`；将 `conversations` 改为 **对象** `{ "question", "answer" }`（按 `human` → `gpt` 成对抽取；去掉固定 system 提示语）。**多轮对话整条跳过，不写入 MongoDB。**  
3. 写入数据库 `mem0_agent_memory`，集合 `genshin_wiki`  
4. **集合与索引**：若集合不存在则 `create_collection`；按名称检测并补建 Atlas **vectorSearch** 索引 `vector_index`（`conversations.answer` 为 `autoEmbed` + `page_name` / `component_name` 为 filter）与 **Search** 全文索引 `answer_text_index`（`conversations.answer` 为 `string` + `lucene.standard`）。需 MongoDB Atlas 与兼容的集群版本

需配置环境变量 `MONGODB_URI`（与项目 `config.py` 一致）。

In [5]:
import json
import sys
from pathlib import Path

# 定位 backend（本 notebook 与 config.py 同目录）；与 config 一致加载 agent_memory_demo/.env
backend_dir = Path.cwd().resolve()
if not (backend_dir / "config.py").is_file():
    for p in [backend_dir, *backend_dir.parents]:
        if (p / "config.py").is_file():
            backend_dir = p
            break
    else:
        raise RuntimeError(
            "找不到 config.py：请在 mem0_memory_gaming_app/backend 下运行本 notebook。"
        )

sys.path.insert(0, str(backend_dir))

from dotenv import load_dotenv

for env_path in (
    backend_dir.parents[1] / ".env",  # agent_memory_demo/.env，与 config.py parents[2] 一致
    backend_dir.parent / ".env",
    Path(".env"),
):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

from config import MONGODB_URI

JSON_PATH = Path("/Users/binzhou/Downloads/my-genshin-data/genshin_knowledge_base.json")
MONGODB_DB = "mem0_agent_memory"
MONGODB_COLLECTION = "game_wiki"
VECTOR_INDEX_NAME = "vector_search_index"
FULLTEXT_INDEX_NAME = "text_search_index"
BATCH_SIZE = 500

In [8]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel

client = MongoClient(MONGODB_URI)
db = client[MONGODB_DB]

# 初始化集合、向量索引与全文索引
def ensure_collection_and_search_indexes() -> None:
    """创建集合（若不存在）；按需建立 vectorSearch 与 Atlas Search 全文索引。"""
    existing_collections = db.list_collection_names()
    if MONGODB_COLLECTION not in existing_collections:
        db.create_collection(MONGODB_COLLECTION)
    col = db[MONGODB_COLLECTION]
    
    # 对conversations.answer 创建向量索引
    if not list(col.list_search_indexes(name=VECTOR_INDEX_NAME)):
        vector_definition = {
            "fields": [
                {
                    "type": "autoEmbed",
                    "modality": "text",
                    "path": "answer",
                    "model": "voyage-4",
                },
                {"type": "filter", "path": "category"},
              
            ]
        }
        col.create_search_index(
            SearchIndexModel(
                definition=vector_definition,
                name=VECTOR_INDEX_NAME,
                type="vectorSearch",
            )
        )

    # 对answer和question创建全文索引
    if not list(col.list_search_indexes(name=FULLTEXT_INDEX_NAME)):
        text_definition = {
            "mappings": {
                "dynamic": False,
                "fields": {
                    "question": {
                        "type": "string",
                        "analyzer": "lucene.standard",
                    },
                    "answer": {
                        "type": "string",
                        "analyzer": "lucene.standard",
                    }
                },
            }
        }
        col.create_search_index(
            SearchIndexModel(
                definition=text_definition,
                name=FULLTEXT_INDEX_NAME,
                type="search",
            )
        )

ensure_collection_and_search_indexes()
col = db[MONGODB_COLLECTION]

# 如需全量重导，取消下一行注释以清空集合
# col.delete_many({})

# inserted = 0
# for i in range(0, len(documents), BATCH_SIZE):
#     batch = documents[i : i + BATCH_SIZE]
#     r = col.insert_many(batch)
#     inserted += len(r.inserted_ids)

# inserted, col.count_documents({})

In [ ]:
sample = col.find_one({})
client.close()
sample